# Coffee17 preprocessing — ALL-IN-ONE Kaggle

Satu notebook, satu **Run All**: provenance/audit → R0/C0/F0/W0 (5 fold masing-masing = 20 primary runs) → validation confirmation → one-time OOF → bootstrap/class analysis/efficiency → final report.

**Kaggle only.** Tidak memakai Google Drive atau Hugging Face sebagai penyimpanan/checkpoint service.

Sebelum Run All:
1. Add Input: **Coffee Green Bean with 17 Defects Original**.
2. Aktifkan GPU.
3. Aktifkan Internet untuk clone GitHub dan instalasi paket.
4. Klik **Run All**.

Jika sebelumnya pernah terputus dan kamu sudah menyimpan output version, tambahkan output notebook ini sebagai Input. Pipeline akan me-reuse checkpoint yang kontraknya identik dan melanjutkan.


In [ ]:
# ONE-RUN KAGGLE PIPELINE: setup -> audit -> 20 primary runs -> decision -> OOF -> analysis
CODE_COMMIT='7de2abb46efd5e71dbab508e2ae4b4e61102a7aa'
REFERENCE_COMMIT='6ef389c23932e44fe4135c32d471b3008b1cbf39'

import hashlib, importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
PROJECT=WORK/'coffee17-preprocessing-project'
REPO=WORK/'coffee-bean-classification-code'
DET=WORK/'coffee-bean-detection-reference'

assert INPUT.is_dir() and WORK.is_dir(), 'Notebook ini harus dijalankan di Kaggle.'

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def merge_tree_exact(source, target):
    source=Path(source); target=Path(target)
    for item in sorted(source.rglob('*')):
        if not item.is_file():
            continue
        rel=item.relative_to(source)
        dst=target/rel
        dst.parent.mkdir(parents=True,exist_ok=True)
        if dst.is_file():
            if sha256_file(item)!=sha256_file(dst):
                raise RuntimeError(f'Kaggle input conflict: {rel}')
        else:
            shutil.copy2(item,dst)

def run(command, cwd=None):
    print('\n$', ' '.join(map(str,command)), flush=True)
    subprocess.run([str(x) for x in command],cwd=cwd,check=True)

# Optional resume: add a previous output of THIS SAME notebook as Kaggle Input.
prior_projects=sorted(
    p for p in INPUT.rglob('coffee17-preprocessing-project')
    if p.is_dir()
)
for prior in prior_projects:
    print('MERGE PRIOR KAGGLE OUTPUT:',prior)
    merge_tree_exact(prior,PROJECT)

# Exact frozen experiment code.
if REPO.exists():
    shutil.rmtree(REPO)
run(['git','clone','--quiet','--no-checkout',
     'https://github.com/ediprin/coffee-bean-classification.git',REPO])
run(['git','-C',REPO,'checkout','--quiet','--detach',CODE_COMMIT])

prior_lock=PROJECT/'evidence/coffee17-preprocessing-runtime-v1/requirements_preprocessing_study_lock.txt'
requirements=prior_lock if prior_lock.is_file() else REPO/'requirements/preprocessing-study.txt'
run([sys.executable,'-m','pip','install','-q','-r',requirements])
run([sys.executable,'-m','pip','install','-q','--no-deps','-e',REPO])
sys.path.insert(0,str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Aktifkan Kaggle GPU sebelum Run All.')
print('GPU:',torch.cuda.get_device_name(0))

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import materialize_preprocessing_development
from bilinear_lmmd.core.run_lock import exclusive_training_lock
from bilinear_lmmd.experiments.preprocessing_environment import freeze_environment, verify_environment
from bilinear_lmmd.experiments.run_preprocessing_static_preflight import run_preprocessing_static_preflight
from bilinear_lmmd.experiments.run_preprocessing_observability_audit import run_observability_audit
from bilinear_lmmd.experiments.verify_preprocessing_reference_equivalence import verify_reference_equivalence
from bilinear_lmmd.experiments.run_preprocessing_primary_confirmation import build_primary_confirmation
from bilinear_lmmd.experiments.run_preprocessing_oof import run_oof

print('\n=== 1/8 COFFEE17 KAGGLE INPUT + PROVENANCE ===')
by_class=discover_directory_samples(INPUT)
print('Coffee17 mounted:',sum(len(v) for v in by_class.values()),'images /',len(by_class),'classes')

ARCHIVE=WORK/'coffee17_original.zip'
if ARCHIVE.exists():
    ARCHIVE.unlink()
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info=zipfile.ZipInfo(
                f'{class_name}/{path.name}',
                date_time=(1980,1,1,0,0,0)
            )
            info.compress_type=zipfile.ZIP_STORED
            info.external_attr=0o644 << 16
            bundle.writestr(info,path.read_bytes())

PROV_LOCAL=WORK/'coffee17_provenance'
CANONICAL=WORK/'coffee17_original_v1'
FOLDS_LOCAL=WORK/'coffee17_folds'
for path in (PROV_LOCAL,CANONICAL,FOLDS_LOCAL):
    if path.exists():
        shutil.rmtree(path)

provenance=audit_coffee17_provenance(
    ARCHIVE,PROV_LOCAL,canonical_root=CANONICAL
)
if provenance['decision']!='PASS':
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary=prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL/'coffee17_provenance.json',
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary['decision']!='PASS_COFFEE17_PREPROCESSING_DATA_GATE':
    raise RuntimeError('Data gate gagal')

print('\n=== 2/8 FREEZE/VERIFY RUNTIME + STATIC + OBSERVABILITY ===')
DATA_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-data-v1'
RUNTIME_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-runtime-v1'
STATIC_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-static-v2'
OBS_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-observability-v1'
REF_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-reference-v2'
LOCKS=PROJECT/'locks/coffee17-preprocessing-v1'
for directory in (
    DATA_EVIDENCE,RUNTIME_EVIDENCE,STATIC_EVIDENCE,
    OBS_EVIDENCE,REF_EVIDENCE,LOCKS
):
    directory.mkdir(parents=True,exist_ok=True)

def persist_exact(source,target):
    source=Path(source); target=Path(target)
    if target.is_file():
        if sha256_file(source)!=sha256_file(target):
            raise RuntimeError(f'Persistent evidence berbeda: {target}')
    else:
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(source,target)

with exclusive_training_lock(
    LOCKS,lock_name='common_setup.lock',stale_seconds=900
):
    persist_exact(
        PROV_LOCAL/'coffee17_provenance.json',
        DATA_EVIDENCE/'coffee17_provenance.json'
    )
    persist_exact(
        PROV_LOCAL/'coffee17_raw_manifest.json',
        DATA_EVIDENCE/'coffee17_raw_manifest.json'
    )
    for name in ('clean_manifest.json','fold_manifest.json','fold_summary.json'):
        persist_exact(FOLDS_LOCAL/name,DATA_EVIDENCE/name)

    ENV=RUNTIME_EVIDENCE/'runtime_environment.json'
    LOCK=RUNTIME_EVIDENCE/'requirements_preprocessing_study_lock.txt'
    if ENV.is_file():
        verify_environment(ENV)
    else:
        freeze_environment(ENV,LOCK,authorize_freeze=True)

    STATIC=STATIC_EVIDENCE/'static_preflight.json'
    if STATIC.is_file():
        static_existing=json.loads(STATIC.read_text())
        if static_existing.get('decision')!='PASS_PREPROCESSING_STATIC_CONTRACT':
            raise RuntimeError('Static evidence belum PASS')
    else:
        LOCAL_STATIC=WORK/'static_preflight.json'
        run_preprocessing_static_preflight(
            LOCAL_STATIC,seed=42,probe_size=64
        )
        persist_exact(LOCAL_STATIC,STATIC)

    OBS=OBS_EVIDENCE/'preprocessing_observability.json'
    if OBS.is_file():
        obs_existing=json.loads(OBS.read_text())
        if obs_existing.get('decision')!='PASS_PREPROCESSING_OBSERVABILITY_AUDIT':
            raise RuntimeError('Observability evidence belum PASS')
    else:
        LOCAL_OBS=WORK/'preprocessing_observability'
        if LOCAL_OBS.exists():
            shutil.rmtree(LOCAL_OBS)
        run_observability_audit(
            CANONICAL,
            PROV_LOCAL/'coffee17_raw_manifest.json',
            LOCAL_OBS,
            device_name='cuda:0',
            image_size=224,
            expected_count=979,
        )
        persist_exact(
            LOCAL_OBS/'preprocessing_observability.json',OBS
        )
        persist_exact(
            LOCAL_OBS/'preprocessing_observability_per_image.csv',
            OBS_EVIDENCE/'preprocessing_observability_per_image.csv'
        )

verify_environment(ENV)
obs=json.loads(OBS.read_text())
print('OBS DECISION:',obs['decision'])
print('OBS GATES:',json.dumps(obs['gates'],indent=2))
if obs['decision']!='PASS_PREPROCESSING_OBSERVABILITY_AUDIT':
    raise RuntimeError('STOP: observability audit tidak PASS. Training tidak dijalankan.')

print('\n=== 3/8 F0 REFERENCE EQUIVALENCE ===')
if DET.exists():
    shutil.rmtree(DET)
run(['git','clone','--quiet',
     'https://github.com/ediprin/coffee-bean-detection.git',DET])
run(['git','-C',DET,'checkout','--quiet','--force',REFERENCE_COMMIT])
LOCAL_EQ=WORK/'F0_luminance_reference_equivalence.json'
EQUIVALENCE=REF_EVIDENCE/'F0_luminance_reference_equivalence.json'
verify_reference_equivalence(
    'F0',DET,LOCAL_EQ,
    expected_reference_commit=REFERENCE_COMMIT
)
persist_exact(LOCAL_EQ,EQUIVALENCE)

print('\n=== 4/8 PRIMARY TRAINING: 4 ARMS x 5 FOLDS ===')
OUT=PROJECT/'experiments/coffee17-preprocessing-primary-v1'
OUT.mkdir(parents=True,exist_ok=True)

for ARM in ('R0','C0','F0','W0'):
    print(f'\n######## ARM {ARM} ########',flush=True)
    arm_equivalence=EQUIVALENCE if ARM=='F0' else None
    for FOLD in range(1,6):
        DEV=WORK/f'coffee17_dev_fold_{FOLD}'
        if DEV.exists():
            shutil.rmtree(DEV)
        materialize_preprocessing_development(
            CANONICAL,
            FOLDS_LOCAL/'clean_manifest.json',
            FOLDS_LOCAL/'fold_manifest.json',
            DEV,
            fold=FOLD,
        )
        CONTRACT=DEV/'development_contract.json'
        LOG=OUT/f'{ARM}_fold{FOLD}_seed42_run.log'
        command=[
            sys.executable,'-u','-m',
            'bilinear_lmmd.experiments.run_preprocessing_arm',
            '--arm',ARM,'--fold',str(FOLD),'--seed','42',
            '--data-root',str(DEV),
            '--development-contract',str(CONTRACT),
            '--static-preflight',str(STATIC),
            '--observability-audit',str(OBS),
            '--environment',str(ENV),
            '--output-root',str(OUT),
            '--required-commit',CODE_COMMIT,
            '--device','cuda:0',
            '--authorize-training',
        ]
        if arm_equivalence is not None:
            command += ['--equivalence',str(arm_equivalence)]

        print(f'START/RESUME {ARM} fold {FOLD}/5',flush=True)
        with LOG.open('a',encoding='utf-8') as stream:
            process=subprocess.Popen(
                command,cwd=REPO,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,bufsize=1
            )
            assert process.stdout is not None
            for line in process.stdout:
                print(line,end='')
                stream.write(line)
                stream.flush()
            rc=process.wait()
        if rc:
            raise RuntimeError(f'{ARM} fold {FOLD} gagal: {rc}')

        result_path=OUT/'primary'/ARM/f'fold_{FOLD}'/'seed42'/'result.json'
        result=json.loads(result_path.read_text())
        print(
            f"{ARM} fold {FOLD} DONE | "
            f"Macro-F1={result['metrics']['macro_f1']:.4f} | "
            f"Worst-F1={result['metrics']['worst_class_f1']:.4f}"
        )
        shutil.rmtree(DEV,ignore_errors=True)

print('\n=== 5/8 PRIMARY CONFIRMATION ===')
AUTH_DIR=PROJECT/'evidence/coffee17-preprocessing-primary-v1'
AUTH_DIR.mkdir(parents=True,exist_ok=True)
AUTH=AUTH_DIR/'preprocessing_primary_confirmation.json'
confirmation=build_primary_confirmation(
    OUT,
    DATA_EVIDENCE/'clean_manifest.json',
    DATA_EVIDENCE/'fold_manifest.json',
    AUTH,
)
print(
    'DECISION:',confirmation['decision'],
    '| COMPLETED:',confirmation['completed_runs'],
    '| TEST ACCESSED:',confirmation['test_images_accessed']
)
if confirmation['decision']!='AUTHORIZE_OOF_TEST_EVALUATION':
    raise RuntimeError('Primary confirmation tidak mengotorisasi OOF.')

print('\n=== 6/8 ONE-TIME OOF OUTER-TEST INFERENCE ===')
OOF=PROJECT/'oof/coffee17-preprocessing-primary-v1'
oof_result=run_oof(
    canonical_root=CANONICAL,
    clean_manifest=DATA_EVIDENCE/'clean_manifest.json',
    fold_manifest=DATA_EVIDENCE/'fold_manifest.json',
    authority_path=AUTH,
    experiments_root=OUT,
    output_root=OOF,
    authorize_test=True,
)
print('OOF COMPLETE:',oof_result)

print('\n=== 7/8 BOOTSTRAP + CLASS ANALYSIS + EFFICIENCY ===')
MERGED=OOF/'merged'
MASTER=MERGED/'primary_oof_table.csv'
SUMMARY=MERGED/'primary_oof_summary.json'
if not MASTER.is_file() or not SUMMARY.is_file():
    raise FileNotFoundError('OOF merged evidence belum lengkap.')

ANALYSIS=PROJECT/'analysis/coffee17-preprocessing-primary-v1'
ANALYSIS.mkdir(parents=True,exist_ok=True)

run([
    sys.executable,'-u','-m',
    'bilinear_lmmd.experiments.run_preprocessing_bootstrap',
    '--master-table',MASTER,
    '--output',ANALYSIS/'paired_bootstrap.json',
    '--iterations','10000',
],cwd=REPO)
run([
    sys.executable,'-u','-m',
    'bilinear_lmmd.experiments.run_preprocessing_analysis',
    '--master-table',MASTER,
    '--output-dir',ANALYSIS,
],cwd=REPO)
run([
    sys.executable,'-u','-m',
    'bilinear_lmmd.experiments.run_preprocessing_efficiency',
    '--canonical-root',CANONICAL,
    '--clean-manifest',DATA_EVIDENCE/'clean_manifest.json',
    '--output',ANALYSIS/'preprocessing_efficiency.json',
    '--batch-sizes','1','16',
    '--warmup','10',
    '--iterations','50',
],cwd=REPO)
run([
    sys.executable,'-u','-m',
    'bilinear_lmmd.experiments.run_preprocessing_final_report',
    '--oof-summary',SUMMARY,
    '--bootstrap',ANALYSIS/'paired_bootstrap.json',
    '--analysis-summary',ANALYSIS/'analysis_summary.json',
    '--efficiency',ANALYSIS/'preprocessing_efficiency.json',
    '--output-dir',ANALYSIS,
],cwd=REPO)

print('\n=== 8/8 COMPLETE ===')
FINAL=ANALYSIS/'FINAL_PREPROCESSING_REPORT.md'
assert FINAL.is_file(), f'Final report tidak ditemukan: {FINAL}'
print('FINAL REPORT:',FINAL)
print('PROJECT OUTPUT:',PROJECT)
print('20 primary runs + OOF + final analysis selesai.')

# Keep only persistent project outputs in /kaggle/working.
for path in (REPO,DET,PROV_LOCAL,CANONICAL,FOLDS_LOCAL):
    if Path(path).exists():
        shutil.rmtree(path,ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
print('Gunakan Save Version untuk menyimpan /kaggle/working/coffee17-preprocessing-project sebagai Kaggle Notebook Output.')
